In [ ]:
files = {'config.py': '"""Fixed experiment configuration."""\nfrom pathlib import Path\nimport os\nimport random\n\nimport numpy as np\n\nROOT = Path(__file__).resolve().parents[1]\nOUTPUTS = Path(__file__).resolve().parent / "outputs"\nSEED = 42\nN_UNITS = 900\nSPLIT = {"train": 600, "val": 150, "test": 150}\nLIFE_RANGE = (400, 1200)\nGAMMA_SHAPE = 0.8\nRUL_CAP = 150\nCLASSES = ("open", "short", "mouse_bite", "spur", "spurious_copper", "missing_hole")\nBASE_RATE = np.array([0.02, 0.02, 0.05, 0.05, 0.05, 0.03])\nGROWTH = np.array([3.0, 3.0, 1.5, 1.5, 1.5, 2.0])\nPOWER = 2\nSEVERITY = np.array([1.0, 1.0, 0.4, 0.3, 0.4, 0.6])\nUNIT_NOISE = (0, 0.3)\nWINDOWS = (10, 30, 100)\nOBSERVED = (*CLASSES, "total_area", "max_area", "mean_confidence", "weighted_count")\n\n# Five reproducible runs, with the independent member seeds derived from 42.\nMEMBER_SEEDS = (SEED, *(int(s.generate_state(1)[0] % (2**31)) for s in\n                        np.random.SeedSequence(SEED).spawn(4)))\n\n\ndef seed_everything(seed=SEED):\n    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n    random.seed(seed)\n    np.random.seed(seed)\n    import torch\n\n    torch.manual_seed(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.use_deterministic_algorithms(True)\n', 'ensemble.py': '"""Select a validation weight and apply causal smoothing in inspection steps."""\nimport json\n\nimport numpy as np\nfrom scipy.signal import lfilter\n\nfrom config import OUTPUTS, RUL_CAP\nfrom features import load_split, unit_slices\n\n\ndef blend(lightgbm, gru, weight):\n    return weight * lightgbm + (1 - weight) * gru\n\n\ndef smooth(prediction, units, step):\n    raw = np.asarray(prediction, dtype=np.float64) * RUL_CAP\n    result = raw.copy()\n    for rows in unit_slices(units):\n        failure = step[rows] + raw[rows]\n        # The initial EWMA is the first estimate for this unit.\n        averaged, _ = lfilter([0.2], [1.0, -0.8], failure, zi=[0.8 * failure[0]])\n        result[rows] = np.where(raw[rows] < 0.9 * RUL_CAP, averaged - step[rows], raw[rows])\n    return np.clip(result, 0, RUL_CAP)\n\n\ndef choose_weight():\n    val = load_split("val")\n    lightgbm, gru = (np.load(OUTPUTS / f"{name}_val.npy").mean(axis=0) for name in ("lgbm", "gru"))\n    scores = {\n        i / 10: float(np.abs(blend(lightgbm, gru, i / 10) - val["y"]).mean()) for i in range(11)\n    }\n    weight = min(scores, key=scores.__getitem__)\n    combined = smooth(blend(lightgbm, gru, weight), val["unit"], val["step"])\n    with np.load(OUTPUTS / "baselines_val.npz") as baseline:\n        time_only = np.clip(baseline["time_only"] * RUL_CAP, 0, RUL_CAP)\n    target = val["y"] * RUL_CAP\n    report = {\n        "weight_lightgbm": weight,\n        "weight_gru": 1 - weight,\n        "blend_validation_mae_normalized": scores,\n        "final_validation_mae_steps": float(np.abs(combined - target).mean()),\n        "time_only_validation_mae_steps": float(np.abs(time_only - target).mean()),\n    }\n    (OUTPUTS / "ensemble.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n\n\nif __name__ == "__main__":\n    print(json.dumps(choose_weight(), indent=2), flush=True)\n', 'evaluate.py': '"""Open the held-out synthetic test split once, after the validation gate."""\nfrom hashlib import file_digest\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom config import MEMBER_SEEDS, OUTPUTS, ROOT, RUL_CAP, SEED, seed_everything\nfrom ensemble import blend, smooth\nfrom features import N_WINDOW_FEATURES, build, load_split\nfrom train_gru import Histories, load_model, predict, training_device\n\nos.environ.setdefault("MPLCONFIGDIR", str(ROOT / "data/runtime/matplotlib"))\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\ndef metrics(prediction, truth):\n    residual = prediction - truth\n    nasa = np.expm1(np.where(residual < 0, -residual / 13, residual / 10))\n    mae = np.abs(residual).mean()\n    return {\n        "mae_steps": float(mae),\n        "rmse_steps": float(np.sqrt(np.mean(residual**2))),\n        "nasa_sum": float(nasa.sum()),\n        "nasa_mean": float(nasa.mean()),\n        "mae_percent_of_cap": float(100 * mae / RUL_CAP),\n    }\n\n\ndef plots(data, prediction):\n    unit, step, true = (data[key] for key in ("unit", "step", "true_rul"))\n    capped = np.minimum(true, RUL_CAP)\n    units = np.random.default_rng(SEED).choice(np.unique(unit), 5, replace=False)\n    figure, axes = plt.subplots(5, 1, figsize=(10, 12), layout="constrained")\n    for equipment, ax in zip(sorted(units), axes, strict=True):\n        mask = unit == equipment\n        ax.plot(step[mask], true[mask], color="0.7", label="True RUL")\n        ax.plot(step[mask], capped[mask], linestyle="--", label="Capped target")\n        ax.plot(step[mask], prediction[mask], label="Predicted RUL")\n        ax.set(title=f"Unit {equipment}", ylabel="Steps left", xlabel="Elapsed inspection steps")\n    axes[0].legend()\n    figure.savefig(OUTPUTS / "five_test_units.png", dpi=150)\n    plt.close(figure)\n\n    fraction = step / (step + true)\n    stage = np.minimum(9, (10 * fraction).astype(int))\n    figure, ax = plt.subplots(layout="constrained")\n    for target, label in ((capped, "Capped target"), (true, "Uncapped true RUL")):\n        mae = [np.abs(prediction[stage == i] - target[stage == i]).mean() for i in range(10)]\n        ax.plot(np.arange(5, 100, 10), mae, marker="o", label=label)\n    ax.set(xlabel="Percent of simulated life elapsed", ylabel="Mean absolute error (steps)")\n    ax.legend()\n    figure.savefig(OUTPUTS / "error_by_life_stage.png", dpi=150)\n    plt.close(figure)\n\n    figure, axes = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")\n    for ax, target, title in zip(axes, (true, capped), ("Uncapped true RUL", "Capped target"), strict=True):\n        ax.scatter(target, prediction, s=2, alpha=0.1)\n        ax.plot([0, RUL_CAP], [0, RUL_CAP], color="black", linestyle="--")\n        ax.set(title=title, xlabel="True steps left", ylabel="Predicted steps left")\n    figure.savefig(OUTPUTS / "predicted_vs_true.png", dpi=150)\n    plt.close(figure)\n\n\ndef main():\n    seed_everything()\n    torch.set_num_threads(5)\n    gate = subprocess.run([sys.executable, str(Path(__file__).with_name("tests.py"))])\n    if gate.returncode:\n        raise SystemExit("Stopped before opening the test set. See the validation check above.")\n\n    # Fingerprint the frozen model artifacts before reading a test observation.\n    selection = json.loads((OUTPUTS / "ensemble.json").read_text())\n    weight = selection["weight_lightgbm"]\n    frozen = {}\n    artifacts = ["scaling.npz", "baselines.npz", "time_only.txt", "lightgbm_trials.json",\n                 "gru_training.json", "ensemble.json"]\n    artifacts.extend(f"lgbm_{i}.txt" for i in range(len(MEMBER_SEEDS)))\n    artifacts.extend(f"gru_{i}.pt" for i in range(len(MEMBER_SEEDS)))\n    for name in artifacts:\n        with (OUTPUTS / name).open("rb") as file:\n            frozen[name] = file_digest(file, "sha256").hexdigest()\n    with (OUTPUTS / "test_access.json").open("x") as file:\n        json.dump({"validation": selection, "seed_members": MEMBER_SEEDS, "sha256": frozen}, file, indent=2)\n\n    data = load_split("test")\n    x = build("test", data)\n    device = training_device()\n    with np.load(OUTPUTS / "scaling.npz") as stats:\n        history = Histories(data, stats, device)\n    lgbm = np.stack([lgb.Booster(model_file=str(OUTPUTS / f"lgbm_{i}.txt")).predict(x)\n                     for i in range(len(MEMBER_SEEDS))])\n    gru = np.stack([predict(load_model(i, device), history) for i in range(len(MEMBER_SEEDS))])\n    with np.load(OUTPUTS / "baselines.npz") as baseline:\n        mean = np.full(len(x), baseline["mean"])\n        ridge = x[:, :N_WINDOW_FEATURES] @ baseline["ridge_coef"] + baseline["ridge_intercept"]\n    time = lgb.Booster(model_file=str(OUTPUTS / "time_only.txt")).predict(data["step"][:, None])\n\n    def steps(values):\n        return np.clip(np.asarray(values, dtype=np.float64) * RUL_CAP, 0, RUL_CAP)\n\n    def final(first, second):\n        return smooth(blend(first, second, weight), data["unit"], data["step"])\n\n    model_predictions = {\n        "Train mean": (steps(mean), None),\n        "Time-only LightGBM": (steps(time), None),\n        "Ridge": (steps(ridge), None),\n        "LightGBM": (steps(lgbm.mean(axis=0)), steps(lgbm)),\n        "GRU": (steps(gru.mean(axis=0)), steps(gru)),\n        "Final ensemble": (final(lgbm.mean(axis=0), gru.mean(axis=0)),\n                           np.stack([final(a, b) for a, b in zip(lgbm, gru, strict=True)])),\n    }\n    true = data["true_rul"].astype(np.float64)\n    near_failure = true <= 50\n    rows, seed_rows = [], []\n    for name, (prediction, members) in model_predictions.items():\n        for subset, mask in (("all", np.ones(len(true), dtype=bool)), ("true RUL <= 50", near_failure)):\n            for target, truth in (("true", true), ("capped", np.minimum(true, RUL_CAP))):\n                result = {"model": name, "subset": subset, "target": target, "n": int(mask.sum()),\n                          **metrics(prediction[mask], truth[mask])}\n                if members is not None:\n                    scores = pd.DataFrame([metrics(p[mask], truth[mask]) for p in members])\n                    for key in scores:\n                        result[f"{key}_seed_mean"] = float(scores[key].mean())\n                        result[f"{key}_seed_std"] = float(scores[key].std(ddof=1))\n                    for seed, row in zip(MEMBER_SEEDS, scores.to_dict("records"), strict=True):\n                        seed_rows.append({"model": name, "subset": subset, "target": target, "seed": seed, **row})\n                rows.append(result)\n    frame = pd.DataFrame(rows)\n    frame.to_csv(OUTPUTS / "test_comparison.csv", index=False)\n    pd.DataFrame(seed_rows).to_csv(OUTPUTS / "test_by_seed.csv", index=False)\n    plots(data, model_predictions["Final ensemble"][0])\n    (OUTPUTS / "test_report.json").write_text(json.dumps(\n        {"synthetic": True, "test_units": int(len(np.unique(data["unit"]))), "validation": selection,\n         "results": rows}, indent=2, allow_nan=False,\n    ) + "\\n")\n    print(frame[["model", "subset", "target", "mae_steps", "rmse_steps", "nasa_sum", "nasa_mean",\n                 "mae_percent_of_cap", "mae_steps_seed_mean", "mae_steps_seed_std"]].to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n', 'features.py': '"""Build causal features with scaling fitted only on training units."""\nimport argparse\n\nimport numpy as np\nfrom sklearn.preprocessing import StandardScaler\n\nfrom config import OBSERVED, OUTPUTS, WINDOWS\n\nN_WINDOW_FEATURES = len(WINDOWS) * len(OBSERVED) * 3\nN_FEATURES = N_WINDOW_FEATURES + len(OBSERVED) + 7 + 1\n\n\ndef unit_slices(units):\n    edges = np.r_[0, np.flatnonzero(np.diff(units)) + 1, len(units)]\n    return (slice(a, b) for a, b in zip(edges[:-1], edges[1:], strict=True))\n\n\ndef unit_features(values, steps):\n    values = np.asarray(values, dtype=np.float64)\n    steps = np.asarray(steps, dtype=np.float64)\n    end = np.arange(1, len(values) + 1)\n    zeros = np.zeros((1, values.shape[1]))\n\n    def prefix(array):\n        return np.vstack((zeros, np.cumsum(array, axis=0)))\n\n    sums = prefix(values)\n    squares = prefix(values**2)\n    time_products = prefix(values * steps[:, None])\n    columns = []\n    for width in WINDOWS:\n        start = np.maximum(0, end - width)\n        size = (end - start)[:, None]\n        count_sum = sums[end] - sums[start]\n        mean = count_sum / size\n        std = np.sqrt(np.maximum(0, (squares[end] - squares[start]) / size - mean**2))\n        center = ((start + end - 1) / 2)[:, None]\n        slope = (time_products[end] - time_products[start] - center * count_sum) / np.maximum(\n            size * (size**2 - 1) / 12, 1,\n        )\n        columns.extend((mean, std, slope))\n    ewma = np.empty_like(values)\n    ewma[0] = values[0]\n    for i in range(1, len(values)):\n        ewma[i] = 0.05 * values[i] + 0.95 * ewma[i - 1]\n    return np.column_stack((*columns, ewma, sums[1:, [0, 1, 2, 3, 4, 5, 9]], steps))\n\n\ndef load_split(split):\n    with np.load(OUTPUTS / f"{split}.npz") as source:\n        return dict(source)\n\n\ndef build(split, data=None):\n    if data is None:\n        data = load_split(split)\n    path = OUTPUTS / f"{split}_features.npy"\n    matrix = np.lib.format.open_memmap(path, mode="w+", dtype=np.float32,\n                                       shape=(len(data["step"]), N_FEATURES))\n    for rows in unit_slices(data["unit"]):\n        matrix[rows] = unit_features(data["observed"][rows], data["step"][rows])\n\n    if split == "train":\n        tabular = StandardScaler().fit(matrix)\n        sequence = StandardScaler().fit(data["observed"])\n        np.savez(OUTPUTS / "scaling.npz", tabular_mean=tabular.mean_, tabular_scale=tabular.scale_,\n                 sequence_mean=sequence.mean_, sequence_scale=sequence.scale_)\n    with np.load(OUTPUTS / "scaling.npz") as stats:\n        mean, scale = stats["tabular_mean"], stats["tabular_scale"]\n        for start in range(0, len(matrix), 16_384):\n            rows = matrix[start:start + 16_384]\n            rows[:] = (rows - mean) / scale\n    matrix.flush()\n    return matrix\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.parse_args()\n    for split in ("train", "val"):\n        print(f"{split} features: {build(split).shape}", flush=True)\n', 'simulate.py': '"""Calibrate the checkpoint once, then generate split synthetic trajectories."""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport sys\nimport xml.etree.ElementTree as ET\n\nimport numpy as np\n\nfrom config import (\n    BASE_RATE, CLASSES, GAMMA_SHAPE, GROWTH, LIFE_RANGE, N_UNITS, OUTPUTS, POWER,\n    ROOT, RUL_CAP, SEED, SEVERITY, SPLIT, UNIT_NOISE,\n)\n\n\ndef calibrate():\n    sys.path.insert(0, str(ROOT / "src"))\n    os.environ.setdefault("YOLO_CONFIG_DIR", str(ROOT / "data/runtime/ultralytics"))\n    os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "data/runtime/matplotlib"))\n    import cv2\n    import torch\n    from ultralytics import YOLO\n    from backend.mirror_merge import merge_mirror_boxes\n    from backend.model_registry import PROJECT_MODEL\n    from config import seed_everything\n\n    seed_everything()\n    torch.set_num_threads(4)\n    folder = ROOT / "artifacts/models" / PROJECT_MODEL\n    info = json.loads((folder / "model_info.json").read_text())\n    weights = folder / "model.pt"\n    digest = hashlib.sha256(weights.read_bytes()).hexdigest()\n    if digest != info["checkpoint_sha256"]:\n        raise ValueError("Detector checkpoint checksum mismatch")\n    dataset = ROOT / "data/raw/pku/PCB_DATASET"\n    with (ROOT / "reports/revamp/pcb100_defectdet_missing_hole_layout11_verification.json").open() as file:\n        holdout = json.load(file)["splits"]["pku_layout_holdout"]\n    groups = sorted(holdout["candidate_by_group"])\n    paths = sorted(path for path in (dataset / "images").rglob("*.jpg") if path.stem[:2] in groups)\n    if len(paths) != holdout["images"]:\n        raise ValueError("Could not reconstruct the detector\'s published PKU holdout")\n\n    canonical = {"open_circuit": "open", "mousebite": "mouse_bite"}\n    model = YOLO(str(weights))\n    detector_classes = [canonical.get(name, name) for name in model.names.values()]\n    if set(detector_classes) != set(CLASSES):\n        raise ValueError("Unexpected checkpoint classes")\n    infer = info["inference"]\n    options = dict(imgsz=info["image_size"], conf=0.001, iou=infer["iou"], rect=infer["rect"],\n                   max_det=infer["max_det"], agnostic_nms=infer["agnostic_nms"],\n                   augment=infer["augment"], device="cpu", verbose=False, save=False)\n    confidence = 0.25\n    totals = {name: {"ground_truth": 0, "tp": 0, "fp": 0} for name in CLASSES}\n    validation_digest = hashlib.sha256()\n    for path in paths:\n        image_bytes = path.read_bytes()\n        xml = (dataset / "Annotations" / path.parent.name / path.with_suffix(".xml").name).read_bytes()\n        validation_digest.update(hashlib.sha256(image_bytes).digest())\n        validation_digest.update(hashlib.sha256(xml).digest())\n        image = cv2.imdecode(np.frombuffer(image_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)\n        width = image.shape[1]\n        original = model.predict(image, **options)[0].boxes.data.cpu()\n        reflected = model.predict(cv2.flip(image, 1), **{**options, "imgsz": infer["mirror_image_size"]})[0]\n        boxes = merge_mirror_boxes(original, reflected.boxes.data.cpu(), width, options["iou"], options["max_det"])\n        truth = {name: [] for name in CLASSES}\n        for item in ET.fromstring(xml).findall("object"):\n            name = item.findtext("name")\n            truth[canonical.get(name, name)].append(\n                [float(item.findtext("bndbox/" + key)) for key in ("xmin", "ymin", "xmax", "ymax")],\n            )\n        for class_id, name in enumerate(detector_classes):\n            predictions = boxes[(boxes[:, 5] == class_id) & (boxes[:, 4] >= confidence), :5].numpy()\n            tp, fp = match(predictions, truth[name])\n            totals[name]["ground_truth"] += len(truth[name])\n            totals[name]["tp"] += tp\n            totals[name]["fp"] += fp\n    if any(value["ground_truth"] == 0 for value in totals.values()):\n        raise ValueError("Every class must have ground-truth boxes")\n    if totals["missing_hole"]["ground_truth"] != holdout["labels"]:\n        raise ValueError("The holdout missing-hole box count changed")\n    stats = {"validation_sha256": validation_digest.hexdigest(), "confidence": confidence, "matching_iou": 0.5, "images": len(paths),\n             "validation": "PKU detector holdout with full six-class annotations",\n             "layouts": groups, "checkpoint_sha256": digest,\n             "inference": "Original and horizontal mirror merged before a global confidence cutoff",\n             "per_class": {name: dict(value, recall=value["tp"] / value["ground_truth"],\n                                      fp_per_image=value["fp"] / len(paths))\n                           for name, value in totals.items()}}\n    OUTPUTS.mkdir(exist_ok=True)\n    (OUTPUTS / "detector_stats.json").write_text(json.dumps(stats, indent=2) + "\\n")\n    return stats\n\n\ndef match(predictions, labels):\n    remaining = np.asarray(labels, dtype=float).reshape(-1, 4)\n    tp = 0\n    for box in sorted(predictions, key=lambda p: -p[4]):\n        if not len(remaining):\n            break\n        intersection = np.maximum(np.minimum(box[2:4], remaining[:, 2:]) -\n                                  np.maximum(box[:2], remaining[:, :2]), 0).prod(axis=1)\n        area = (box[2:4] - box[:2]).prod()\n        union = area + (remaining[:, 2:] - remaining[:, :2]).prod(axis=1) - intersection\n        ious = intersection / np.maximum(union, 1e-12)\n        best = int(ious.argmax())\n        if ious[best] >= 0.5:\n            tp += 1\n            remaining = np.delete(remaining, best, axis=0)\n    return tp, len(predictions) - tp\n\n\ndef simulate_unit(rng, recall, false_positives):\n    life = rng.uniform(*LIFE_RANGE)\n    multiplier = rng.lognormal(*UNIT_NOISE)\n    # Continue drawing increments if the initial block does not reach failure.\n    increments = []\n    total = 0.0\n    while total < 1:\n        block = rng.gamma(GAMMA_SHAPE, 1 / (GAMMA_SHAPE * life), size=int(LIFE_RANGE[1]))\n        increments.append(block)\n        total += block.sum()\n    damage = np.cumsum(np.concatenate(increments))\n    failure = int(np.searchsorted(damage, 1)) + 1\n    damage = np.r_[0.0, damage[:failure]]\n    latent_rate = multiplier * (BASE_RATE + GROWTH * damage[:, None] ** POWER)\n    true = rng.binomial(rng.poisson(latent_rate), recall)\n    false = rng.poisson(false_positives, size=true.shape)\n    counts = true + false\n    true_per_step = true.sum(axis=1)\n    false_per_step = false.sum(axis=1)\n    count = true_per_step + false_per_step\n    index = np.repeat(np.arange(len(count)), count)\n    area = np.exp(rng.normal(-6 + damage[index], 0.5))\n    area_max = np.zeros(len(count))\n    np.maximum.at(area_max, index, area)\n    confidence_sum = np.bincount(np.repeat(np.arange(len(count)), true_per_step),\n                                 rng.beta(8, 3, true_per_step.sum()), minlength=len(count))\n    confidence_sum += np.bincount(np.repeat(np.arange(len(count)), false_per_step),\n                                  rng.beta(3, 6, false_per_step.sum()), minlength=len(count))\n    observed = np.column_stack((counts, np.bincount(index, area, minlength=len(count)), area_max,\n                                confidence_sum / np.maximum(count, 1), counts @ SEVERITY))\n    return observed, failure - np.arange(len(damage)), damage\n\n\ndef assignment():\n    order = np.random.default_rng(SEED).permutation(N_UNITS)\n    stop = np.cumsum(list(SPLIT.values()))[:-1]\n    return dict(zip(SPLIT, np.split(order, stop), strict=True))\n\n\ndef generate(stats):\n    recall = np.array([stats["per_class"][name]["recall"] for name in CLASSES])\n    fp = np.array([stats["per_class"][name]["fp_per_image"] for name in CLASSES])\n    rngs = np.random.SeedSequence(SEED).spawn(N_UNITS)\n    for split, units in assignment().items():\n        observations, rul, auxiliary, ids, steps = [], [], [], [], []\n        for unit in sorted(units):\n            x, remaining, damage = simulate_unit(np.random.default_rng(rngs[unit]), recall, fp)\n            if np.any(np.diff(damage) < 0) or damage[-1] < 1 or np.any(damage[:-1] >= 1):\n                raise ValueError(f"Invalid damage history for unit {unit}")\n            observations.append(x.astype(np.float32))\n            rul.append(remaining.astype(np.int16))\n            auxiliary.append(np.minimum(damage, 1).astype(np.float32))\n            ids.append(np.full(len(remaining), unit, dtype=np.int16))\n            steps.append(np.arange(len(remaining), dtype=np.int16))\n        np.savez_compressed(OUTPUTS / f"{split}.npz", observed=np.concatenate(observations),\n                            unit=np.concatenate(ids), step=np.concatenate(steps),\n                            true_rul=np.concatenate(rul),\n                            y=np.minimum(np.concatenate(rul), RUL_CAP).astype(np.float32) / RUL_CAP,\n                            auxiliary=np.concatenate(auxiliary))\n        print(f"{split}: {len(units)} units, {sum(map(len, steps))} steps", flush=True)\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("stage", choices=("calibrate", "generate"))\n    stage = parser.parse_args().stage\n    if stage == "calibrate":\n        if (OUTPUTS / "detector_stats.json").exists():\n            raise SystemExit("Calibration already exists.")\n        print(json.dumps(calibrate(), indent=2), flush=True)\n    else:\n        generate(json.loads((OUTPUTS / "detector_stats.json").read_text()))\n', 'tests.py': '"""Tests that must pass before opening the held-out test split."""\nimport json\nimport unittest\n\nimport numpy as np\n\nfrom config import N_UNITS, OUTPUTS, SEED, SPLIT, CLASSES, seed_everything\nfrom ensemble import smooth\nfrom features import load_split, unit_features\nfrom simulate import assignment, simulate_unit\nfrom train_gru import GRU, Histories, training_device\nfrom train_lgbm import features, training_params\n\n\ndef detector_rates():\n    stats = json.loads((OUTPUTS / "detector_stats.json").read_text())["per_class"]\n    return (np.array([stats[c][field] for c in CLASSES]) for field in ("recall", "fp_per_image"))\n\n\nclass DataTests(unittest.TestCase):\n    def test_damage_never_decreases_and_stops_at_first_failure(self):\n        recall, fp = detector_rates()\n        # Generation checks every trajectory. This gate rechecks only train/validation units.\n        seeds = np.random.SeedSequence(SEED).spawn(N_UNITS)\n        splits = assignment()\n        for unit in np.r_[splits["train"], splits["val"]]:\n            _, remaining, damage = simulate_unit(np.random.default_rng(seeds[unit]), recall, fp)\n            self.assertTrue(np.all(np.diff(damage) >= 0))\n            self.assertTrue(np.all(damage[:-1] < 1))\n            self.assertGreaterEqual(damage[-1], 1)\n            self.assertEqual(remaining[-1], 0)\n\n    def test_units_are_disjoint(self):\n        splits = assignment()\n        self.assertEqual({key: len(units) for key, units in splits.items()}, SPLIT)\n        self.assertEqual(len(np.unique(np.concatenate(list(splits.values())))), N_UNITS)\n        for name in ("train", "val"):\n            np.testing.assert_array_equal(np.unique(load_split(name)["unit"]), np.sort(splits[name]))\n\n    def test_truncation_does_not_change_past_features(self):\n        values, _, _ = simulate_unit(np.random.default_rng(SEED), *detector_rates())\n        steps = np.arange(len(values))\n        full = unit_features(values, steps)\n        unit = np.zeros_like(steps)\n        prediction = np.random.default_rng(SEED).uniform(size=len(steps))\n        for stop in (1, 10, 30, 100, len(values) // 2):\n            np.testing.assert_array_equal(unit_features(values[:stop], steps[:stop]), full[:stop])\n            np.testing.assert_array_equal(smooth(prediction[:stop], unit[:stop], steps[:stop]),\n                                          smooth(prediction, unit, steps)[:stop])\n        # The sequence view starts at the correct unit and pads only on the right.\n        history = Histories({"observed": values[:110], "step": steps[:110],\n                             "y": np.zeros(110), "auxiliary": np.zeros(110)},\n                            {"sequence_mean": np.zeros(10), "sequence_scale": np.ones(10)})\n        import torch\n        seq, mask = history.batch(torch.tensor([0, 109]))\n        self.assertEqual(mask.sum(dim=1).tolist(), [1, 100])\n        self.assertFalse(torch.any(seq[0, 1:]))\n        np.testing.assert_array_equal(seq[1, :, :10].numpy(), values[10:110].astype(np.float32))\n\n    def test_same_seed_produces_identical_results(self):\n        rates = tuple(detector_rates())\n        outputs = [simulate_unit(np.random.default_rng(SEED), *rates) for _ in range(2)]\n        for first, second in zip(*outputs, strict=True):\n            np.testing.assert_array_equal(first, second)\n        import torch\n        seed_everything()\n        device = training_device()\n        sequences = torch.ones(2, 5, 11, device=device)\n        mask = torch.ones(2, 5, dtype=torch.bool, device=device)\n        predictions = []\n        for _ in range(2):\n            seed_everything()\n            model = GRU().to(device)\n            optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)\n            rul, damage = model(sequences, mask)\n            loss = torch.nn.functional.huber_loss(rul, torch.zeros_like(rul))\n            loss += 0.5 * torch.nn.functional.huber_loss(damage, torch.zeros_like(damage))\n            loss.backward()\n            optimizer.step()\n            predictions.append(model.eval()(sequences, mask)[0].detach().cpu().numpy())\n        np.testing.assert_array_equal(*predictions)\n\n        import lightgbm as lgb\n        # Sample training rows across the entire split, including the near-failure targets.\n        indices = np.random.default_rng(SEED).choice(len(load_split("train")["y"]), 4096, replace=False)\n        x, y = features("train")[indices], load_split("train")["y"][indices]\n        params = {**training_params(), "seed": SEED}\n        predictions = [lgb.train(params, lgb.Dataset(x, label=y), num_boost_round=8).predict(x)\n                       for _ in range(2)]\n        np.testing.assert_array_equal(*predictions)\n\n\nclass ValidationTest(unittest.TestCase):\n    def test_final_model_beats_time_only_on_validation(self):\n        metrics = json.loads((OUTPUTS / "ensemble.json").read_text())\n        self.assertLess(metrics["final_validation_mae_steps"], metrics["time_only_validation_mae_steps"],\n                        "Stop: the final model did not beat the time-only baseline.")\n\n\nif __name__ == "__main__":\n    unittest.main(verbosity=2)\n', 'train_gru.py': '"""Train five deterministic two-head GRUs on synthetic trajectories."""\nimport json\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom config import MEMBER_SEEDS, OBSERVED, OUTPUTS, seed_everything\nfrom features import load_split\n\nSEQUENCE = 100\nBATCH = 256\nEPOCHS = 40\n\n\ndef training_device():\n    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n\n\nclass Histories:\n    def __init__(self, data, stats, device="cpu"):\n        observed = (data["observed"] - stats["sequence_mean"]) / stats["sequence_scale"]\n        self.values = torch.from_numpy(np.column_stack((observed, data["step"] / 1200)).astype(np.float32)).to(device)\n        self.steps = torch.from_numpy(data["step"].astype(np.int64)).to(device)\n        self.y = torch.from_numpy(data["y"]).to(device)\n        self.auxiliary = torch.from_numpy(data["auxiliary"]).to(device)\n        self.offsets = torch.arange(SEQUENCE, device=device)\n\n    def batch(self, indices):\n        lengths = (self.steps[indices] + 1).clamp(max=SEQUENCE)\n        offsets = self.offsets if self.values.is_cuda else self.offsets[:int(lengths.max())]\n        mask = offsets[None, :] < lengths[:, None]\n        rows = indices[:, None] - lengths[:, None] + 1 + offsets[None, :]\n        sequence = self.values[torch.where(mask, rows, 0)]\n        return sequence.masked_fill(~mask[:, :, None], 0), mask\n\n\nclass GRU(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.gru = nn.GRU(len(OBSERVED) + 1, 96, num_layers=2, dropout=0.1, batch_first=True)\n        self.rul = nn.Linear(96, 1)\n        self.damage = nn.Linear(96, 1)\n\n    def forward(self, sequence, mask):\n        states, _ = self.gru(sequence)\n        # Right padding cannot influence the last valid recurrent state.\n        last = states[torch.arange(len(sequence), device=sequence.device), mask.sum(dim=1) - 1]\n        return self.rul(last).squeeze(1), self.damage(last).squeeze(1)\n\n\n@torch.inference_mode()\ndef predict(model, history):\n    model.eval()\n    values = []\n    for indices in torch.arange(len(history.steps), device=history.steps.device).split(BATCH):\n        rul, _ = model(*history.batch(indices))\n        values.append(rul)\n    return torch.cat(values).cpu().numpy()\n\n\ndef load_model(index, device="cpu"):\n    model = GRU()\n    model.load_state_dict(torch.load(OUTPUTS / f"gru_{index}.pt", map_location="cpu", weights_only=True))\n    return model.to(device)\n\n\ndef fit(index, seed, train, val):\n    seed_everything(seed)\n    model = GRU().to(train.steps.device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)\n    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)\n    huber = nn.HuberLoss()\n    selected = torch.where(train.steps.remainder(5) == 0)[0]\n    validation_targets = val.y.cpu().numpy()\n    best, best_epoch, best_predictions = float("inf"), 0, None\n    history = []\n    for epoch in range(1, EPOCHS + 1):\n        model.train()\n        order = selected[torch.randperm(len(selected)).to(selected.device)]\n        for indices in order.split(BATCH):\n            optimizer.zero_grad(set_to_none=True)\n            rul, damage = model(*train.batch(indices))\n            loss = huber(rul, train.y[indices]) + 0.5 * huber(damage, train.auxiliary[indices])\n            loss.backward()\n            optimizer.step()\n        schedule.step()\n        predictions = predict(model, val)\n        mae = float(np.abs(predictions - validation_targets).mean())\n        history.append(mae)\n        print(f"GRU {index + 1}/5, seed {seed}, epoch {epoch}: validation MAE {mae:.6f}", flush=True)\n        if mae < best:\n            best, best_epoch, best_predictions = mae, epoch, predictions\n            torch.save(model.state_dict(), OUTPUTS / f"gru_{index}.pt")\n        if epoch - best_epoch >= 8:\n            break\n    return best_predictions, {"seed": seed, "best_epoch": best_epoch, "validation_mae": history}\n\n\ndef main():\n    torch.set_num_threads(5)\n    seed_everything()\n    device = training_device()\n    print(f"GRU device: {torch.cuda.get_device_name(device) if device.type == \'cuda\' else \'CPU\'}", flush=True)\n    with np.load(OUTPUTS / "scaling.npz") as stats:\n        train, val = (Histories(load_split(split), stats, device) for split in ("train", "val"))\n    predictions, logs = [], []\n    for i, seed in enumerate(MEMBER_SEEDS):\n        values, history = fit(i, seed, train, val)\n        predictions.append(values)\n        logs.append(history)\n    np.save(OUTPUTS / "gru_val.npy", np.stack(predictions))\n    (OUTPUTS / "gru_training.json").write_text(json.dumps(logs, indent=2) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n', 'train_lgbm.py': '"""Fit the required baselines and tune/finalize LightGBM on train/validation."""\nimport json\nimport os\n\nimport lightgbm as lgb\nimport numpy as np\nimport optuna\nfrom sklearn.linear_model import Ridge\n\nfrom config import MEMBER_SEEDS, OUTPUTS, SEED, seed_everything\nfrom features import N_WINDOW_FEATURES, load_split\n\ndef training_params():\n    backend = os.environ.get("PCB_RUL_LIGHTGBM_DEVICE", "cpu")\n    params = {"objective": "huber", "alpha": 0.9, "metric": "mae", "verbosity": -1,\n              "num_threads": 5, "feature_pre_filter": False, "device_type": backend}\n    if backend == "gpu":\n        return {**params, "gpu_use_dp": True, "gpu_device_id": 0}\n    if backend == "cpu":\n        return {**params, "deterministic": True, "force_col_wise": True}\n    raise ValueError(f"Unsupported LightGBM device: {backend}")\n\n\ndef fit(train, val, params, seed):\n    return lgb.train(\n        {**params, "seed": seed}, train, num_boost_round=3000,\n        valid_sets=[val], callbacks=[lgb.early_stopping(100, verbose=False)],\n    )\n\n\ndef features(split):\n    return np.load(OUTPUTS / f"{split}_features.npy", mmap_mode="r")\n\n\ndef main():\n    seed_everything()\n    base = training_params()\n    print(f"LightGBM backend: {base[\'device_type\']}", flush=True)\n    train, val = (load_split(split) for split in ("train", "val"))\n    x_train, x_val = (features(split) for split in ("train", "val"))\n    y_train, y_val = train["y"], val["y"]\n    time_train = lgb.Dataset(train["step"][:, None], label=y_train)\n    time_val = time_train.create_valid(val["step"][:, None], label=y_val)\n    baseline = fit(time_train, time_val, {**base, "objective": "regression_l1", "verbosity": 1}, SEED)\n    baseline.save_model(OUTPUTS / "time_only.txt")\n    ridge = Ridge().fit(x_train[:, :N_WINDOW_FEATURES], y_train)\n    np.savez(OUTPUTS / "baselines.npz", ridge_coef=ridge.coef_, ridge_intercept=ridge.intercept_,\n             mean=y_train.mean())\n    np.savez(OUTPUTS / "baselines_val.npz", time_only=baseline.predict(val["step"][:, None]),\n             ridge=ridge.predict(x_val[:, :N_WINDOW_FEATURES]), mean=np.full(len(y_val), y_train.mean()))\n    print("Baselines saved", flush=True)\n\n    training = lgb.Dataset(x_train, label=y_train, params=base, free_raw_data=False)\n    validation = training.create_valid(x_val, label=y_val)\n\n    def objective(trial):\n        params = {\n            "num_leaves": trial.suggest_int("num_leaves", 15, 127),\n            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),\n            "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),\n            "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),\n            "lambda_l2": trial.suggest_float("lambda_l2", 0, 10),\n        }\n        model = fit(training, validation, {**base, **params}, SEED)\n        return model.best_score["valid_0"]["l1"]\n\n    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))\n    study.optimize(objective, n_trials=40)\n    (OUTPUTS / "lightgbm_trials.json").write_text(json.dumps({\n        "device_type": base["device_type"], "best_params": study.best_params, "validation_mae": study.best_value,\n        "trials": [{"number": trial.number, "params": trial.params, "mae": trial.value}\n                   for trial in study.trials],\n    }, indent=2) + "\\n")\n    predictions = []\n    for i, seed in enumerate(MEMBER_SEEDS):\n        model = fit(training, validation, {**base, **study.best_params}, seed)\n        model.save_model(OUTPUTS / f"lgbm_{i}.txt")\n        predictions.append(model.predict(x_val))\n        print(f"LightGBM {i + 1}/5, seed {seed}, best round {model.best_iteration}", flush=True)\n    np.save(OUTPUTS / "lgbm_val.npy", np.stack(predictions))\n\n\nif __name__ == "__main__":\n    main()\n', 'README.md': "# PCB remaining useful life experiment\n\nRUL labels are generated from a calibrated stochastic degradation simulator. They are not measured field data.\n\nThe experiment requires Python 3.11, NumPy, pandas, scikit-learn, LightGBM, PyTorch, Optuna, Matplotlib and SciPy. Detector calibration also uses the repository's Ultralytics and OpenCV environment and its existing checkpoint. Run from the repository root.\n\n1. `.venv/bin/python pcb_rul/simulate.py calibrate` runs the existing YOLO checkpoint on its 93-image PKU holdout, layouts 06 and 10. It scores all six classes against the original XML boxes at global confidence 0.25 after the checkpoint's original/mirror merge. Matching is one-to-one per class at IoU 0.5. Calibration is saved once in `outputs/detector_stats.json`.\n2. `.venv/bin/python pcb_rul/simulate.py generate` generates the specified 900 units and saves the splits separately. The simulator starts each trajectory at step 0. It uses the given log-area distribution for all observed boxes and uses the two confidence distributions for their respective true or false detection sources. Empty frames have zero total area, max area and mean confidence. The only saved form of hidden damage is the auxiliary training target.\n3. `.venv/bin/python pcb_rul/features.py` builds train and validation features and fits scaling on the train split. The Ridge baseline uses all 90 rolling window features. The time-only baseline is LightGBM with an absolute-error objective.\n4. `.venv/bin/python pcb_rul/train_lgbm.py` trains the baselines, runs the 40 validation trials and fits the five members. `.venv/bin/python pcb_rul/train_gru.py` trains the five GRU members. Simulation, split selection and Optuna use seed 42. The first member uses 42 and the other four seeds are deterministically derived from 42.\n5. `.venv/bin/python pcb_rul/ensemble.py` selects the blend on validation data. The smoother first converts normalized predictions to steps so it can add them to elapsed steps. It applies the causal EWMA when the current raw estimate is below 135 steps, then clips the output to 0–150.\n6. `.venv/bin/python pcb_rul/evaluate.py` runs all checks in `tests.py`, including the validation gate, before it opens the test split. If the final model fails to beat the time-only baseline, it stops there. A marker records the frozen artifacts and prevents repeated test evaluation.\n\nAll generated outputs go under `outputs/`. The test comparison reports both uncapped true RUL and the capped training target. For true RUL above 150, uncapped error necessarily includes the effect of the output cap. Seed means and sample standard deviations describe the five individual fits and five paired blends. The final ensemble row also reports the prediction made after averaging the members of each model family. The fixed baselines each have one reported score.\n\nThe Kaggle T4 notebook sets `PCB_RUL_LIGHTGBM_DEVICE=gpu` to build LightGBM histograms on the GPU using double precision. The GRU uses CUDA. LightGBM's deterministic flag applies to CPU training, so the pre-test checks repeat small seeded fits on the selected devices and compare their predictions exactly.\n", 'outputs/detector_stats.json': '{\n  "validation_sha256": "a38d4e77ed7f41b4dce0c72076c4604792ce47e4780cd200d8d26a52e3e8dba4",\n  "confidence": 0.25,\n  "matching_iou": 0.5,\n  "images": 93,\n  "validation": "PKU detector holdout with full six-class annotations",\n  "layouts": [\n    "06",\n    "10"\n  ],\n  "checkpoint_sha256": "931f4ceaeb6eb5d4b827eeff2dde51e7d14846554dc2eddb062607ede21df072",\n  "inference": "Original and horizontal mirror merged before a global confidence cutoff",\n  "per_class": {\n    "open": {\n      "ground_truth": 80,\n      "tp": 51,\n      "fp": 255,\n      "recall": 0.6375,\n      "fp_per_image": 2.7419354838709675\n    },\n    "short": {\n      "ground_truth": 80,\n      "tp": 55,\n      "fp": 67,\n      "recall": 0.6875,\n      "fp_per_image": 0.7204301075268817\n    },\n    "mouse_bite": {\n      "ground_truth": 78,\n      "tp": 43,\n      "fp": 2,\n      "recall": 0.5512820512820513,\n      "fp_per_image": 0.021505376344086023\n    },\n    "spur": {\n      "ground_truth": 74,\n      "tp": 47,\n      "fp": 34,\n      "recall": 0.6351351351351351,\n      "fp_per_image": 0.3655913978494624\n    },\n    "spurious_copper": {\n      "ground_truth": 81,\n      "tp": 45,\n      "fp": 6,\n      "recall": 0.5555555555555556,\n      "fp_per_image": 0.06451612903225806\n    },\n    "missing_hole": {\n      "ground_truth": 76,\n      "tp": 70,\n      "fp": 118,\n      "recall": 0.9210526315789473,\n      "fp_per_image": 1.2688172043010753\n    }\n  }\n}\n'}

import os
from pathlib import Path
import shutil
import subprocess
import sys

root = Path("/kaggle/working/pcb_rul")
for name, content in files.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)

environment = os.environ.copy()
environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
environment["PCB_RUL_LIGHTGBM_DEVICE"] = "gpu"
environment["MPLCONFIGDIR"] = "/tmp/pcb-rul-matplotlib"
environment["UV_CACHE_DIR"] = "/tmp/pcb-rul-uv-cache"
environment["UV_PYTHON_INSTALL_DIR"] = "/tmp/pcb-rul-python"
venv = Path("/tmp/pcb-rul-env")
uv = shutil.which("uv")
if uv is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True, env=environment)
    uv = shutil.which("uv")
subprocess.run([uv, "venv", "--python", "3.11", str(venv)], check=True, env=environment)
python = str(venv / "bin/python")
dependencies = [
    "numpy==2.4.6", "pandas==3.0.5", "scikit-learn==1.9.0", "lightgbm==4.7.0",
    "torch==2.7.1", "optuna==5.0.0", "matplotlib==3.11.1", "scipy==1.17.1",
]
subprocess.run([uv, "pip", "install", "--python", python, *dependencies], check=True, env=environment)

def run(*args):
    subprocess.run([python, "-u", *args], cwd=root, check=True, env=environment)

run("-c", "import sys, torch; assert sys.version_info[:2] == (3, 11); assert torch.cuda.is_available(), 'Kaggle GPU unavailable'; print(sys.version, torch.__version__, torch.cuda.get_device_name(0), flush=True)")
run("simulate.py", "generate")
run("features.py")
lgbm = subprocess.Popen([python, "-u", "train_lgbm.py"], cwd=root, env=environment)
try:
    gpu_listing = subprocess.check_output(["nvidia-smi", "--list-gpus"], text=True)
    gru_environment = environment.copy()
    if len(gpu_listing.splitlines()) >= 2:
        gru_environment["CUDA_VISIBLE_DEVICES"] = "1"
    subprocess.run([python, "-u", "train_gru.py"], cwd=root, check=True, env=gru_environment)
    if lgbm.wait():
        raise SystemExit("LightGBM training failed.")
finally:
    if lgbm.poll() is None:
        lgbm.terminate()
        lgbm.wait()
run("ensemble.py")
run("evaluate.py")
